In [ ]:
import re
from typing import Mapping
import json
from pathlib import Path


_IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def _quote_identifier(identifier: str) -> str:
    """
    Quote a SQL identifier using backticks.

    Backticks inside the identifier are escaped according to Spark SQL rules.
    """
    if not isinstance(identifier, str):
        raise TypeError("SQL identifiers must be strings.")

    identifier = identifier.strip()

    if not identifier:
        raise ValueError("SQL identifiers cannot be empty.")

    return f"`{identifier.replace('`', '``')}`"


def _quote_string(value: str) -> str:
    """
    Quote a string literal for Spark SQL.
    """
    if not isinstance(value, str):
        raise TypeError("SQL string values must be strings.")

    return "'" + value.replace("'", "''") + "'"

def is_streaming_table(table_path: str) -> bool:
    """
    Determina si una tabla es de tipo streaming basándose en sus TBLPROPERTIES.
    
    Args:
        table_path: Ruta completa de la tabla (catalog.schema.tabla)
    
    Returns:
        True si la tabla es streaming, False en caso contrario.
    """
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES `{table_path}`").collect()
        props_dict = {row['key']: row['value'] for row in props}
        
        # Propiedad que identifica streaming tables creadas por SDP (Spark Declarative Pipelines)
        table_type = props_dict.get('spark.internal.pipelines.tabletype', '')
        
        return table_type.lower() == 'streaming'
    except Exception as e:
        print(f"Error al consultar la tabla: {e}")
        return False

def get_table_type(full_table_name: str) -> str:
    """
    Given a fully qualified table name 'catalog.schema.table',
    returns the table type: TABLE, VIEW, MATERIALIZED VIEW, STREAMING TABLE, etc.
    """
    parts = full_table_name.strip().split(".")
    if len(parts) != 3:
        raise ValueError(f"Expected format 'catalog.schema.table', got: '{full_table_name}'")
    parts = [p.replace("`", "") for p in parts]
    catalog, schema, table = parts
    
    result = spark.sql(f"""
        SELECT table_type 
        FROM {_quote_identifier(catalog)}.information_schema.tables
        WHERE table_schema = {_quote_string(schema)}
          AND table_name = {_quote_string(table)}
    """)
    
    rows = result.collect()
    if not rows:
        raise ValueError(f"Table '{full_table_name}' not found in information_schema.")
    
    return rows[0]["table_type"].replace("_", " ")


def update_table_metadata(
    catalog: str,
    schema: str,
    table: str,
    table_comment: str,
    table_properties: Mapping[str, str],
    column_comments: Mapping[str, str],
) -> None:
    """
    Update a Databricks Unity Catalog table's comment, properties,
    and column comments.

    Args:
        catalog:
            The Unity Catalog catalog name.

        schema:
            The schema name.

        table:
            The table name.

        table_comment:
            The comment to apply to the table.

        table_properties:
            A mapping of table property names to their values.

        column_comments:
            A mapping where keys are column names and values are
            the comments to apply to those columns.

    Raises:
        TypeError:
            If an argument has an invalid type.

        ValueError:
            If a required string is empty or a mapping contains
            invalid keys/values.

    Example:
        update_table_metadata(
            catalog="main",
            schema="sales",
            table="customers",
            table_comment="Contains customer information.",
            table_properties={
                "domain": "sales",
                "data_owner": "data_team",
            },
            column_comments={
                "customer_id": "Unique customer identifier.",
                "email": "Customer email address.",
            },
        )
    """

    # ------------------------------------------------------------------
    # Validate arguments
    # ------------------------------------------------------------------

    for name, value in {
        "catalog": catalog,
        "schema": schema,
        "table": table,
        "table_comment": table_comment,
    }.items():
        if not isinstance(value, str):
            raise TypeError(f"{name} must be a string.")

        if not value.strip():
            raise ValueError(f"{name} cannot be empty.")

    if not isinstance(table_properties, Mapping):
        raise TypeError("table_properties must be a dictionary or mapping.")

    if not isinstance(column_comments, Mapping):
        raise TypeError("column_comments must be a dictionary or mapping.")

    for key, value in table_properties.items():
        if not isinstance(key, str) or not key.strip():
            raise ValueError(
                "All table property names must be non-empty strings."
            )

        if not isinstance(value, str):
            raise TypeError(
                f"Table property '{key}' must have a string value."
            )

    for column_name, comment in column_comments.items():
        if not isinstance(column_name, str) or not column_name.strip():
            raise ValueError(
                "All column names must be non-empty strings."
            )

        if not isinstance(comment, str):
            raise TypeError(
                f"Comment for column '{column_name}' must be a string."
            )

    # ------------------------------------------------------------------
    # Build fully qualified table name
    # ------------------------------------------------------------------

    full_table_name = ".".join(
        [
            (catalog),
            (schema),
            (table),
        ]
    )
    full_table_name = _quote_identifier(full_table_name)

    table_type = get_table_type(full_table_name)

    # ------------------------------------------------------------------
    # Update table comment
    # ------------------------------------------------------------------
    comment_dict = {"comment": _quote_string(table_comment)}
    comment_properties_sql = ", ".join(
                f"{_quote_string(key)} = {_quote_string(value)}"
                for key, value in comment_dict.items()
            )
    """
    COMMENT ON TABLE {full_table_name}
    IS {_quote_string(table_comment)}
    """
    if True:
        spark.sql(
            f"""
            ALTER TABLE {full_table_name}
            SET TBLPROPERTIES ({comment_properties_sql})
            """
        )

    # ------------------------------------------------------------------
    # Update table properties
    # ------------------------------------------------------------------

    if table_properties:
        properties_sql = ", ".join(
            f"{_quote_string(key)} = {_quote_string(value)}"
            for key, value in table_properties.items()
        )

        spark.sql(
            f"""
            ALTER TABLE {full_table_name}
            SET TBLPROPERTIES ({properties_sql})
            """
        )

    # ------------------------------------------------------------------
    # Update column comments
    # ------------------------------------------------------------------

    for column_name, comment in column_comments.items():
        quoted_column_name = _quote_identifier(column_name)

        spark.sql(
            f"""
            ALTER {table_type} {full_table_name}
            ALTER COLUMN {quoted_column_name}
            COMMENT {_quote_string(comment)}
            """
        )

    return None




def read_json(file_path: str) -> dict:
    if not isinstance(file_path, str):
        raise TypeError(
            f"file_path must be a str, got {type(file_path).__name__}"
        )

    path = Path(file_path)

    # Crear el archivo si no existe
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)

        with path.open("w", encoding="utf-8") as file:
            json.dump({}, file, indent=4)

        return {}

    if not path.is_file():
        raise ValueError(f"Path is not a file: {path}")

    # Si el archivo está vacío, devolver {}
    if path.stat().st_size == 0:
        return {}

    with path.open("r", encoding="utf-8") as file:
        content = file.read().strip()

    # Si solo contiene espacios/saltos de línea, devolver {}
    if not content:
        return {}

    return json.loads(content)




In [ ]:
def generate_column_comment(column_metadata: dict) -> str:
    """
    Generate a standardized column comment from metadata.

    Expected structure:

    {
        "description": str,
        "type": str,
        "is_pk": bool,
        "is_fk": bool,
        "quality": {
            "null_percen": ...,
            "unique_percen": ...,
            "referential_integrity": ...  # Only present when is_fk is True
        },
        "nested_structures": {...}  # Optional
    }

    Returns:
        A formatted string containing the column description,
        type, key information, nested structure information,
        and data quality metrics.
    """

    if not isinstance(column_metadata, dict):
        raise TypeError("column_metadata must be a dictionary.")

    # Required fields
    description = column_metadata.get("description", "")
    column_type = column_metadata.get("type", "")
    is_pk = column_metadata.get("is_pk", False)
    is_fk = column_metadata.get("is_fk", False)

    if not isinstance(description, str):
        raise TypeError("'description' must be a string.")

    if not isinstance(column_type, str):
        raise TypeError("'type' must be a string.")

    if not isinstance(is_pk, bool):
        raise TypeError("'is_pk' must be a boolean.")

    if not isinstance(is_fk, bool):
        raise TypeError("'is_fk' must be a boolean.")

    quality = column_metadata.get("quality", {})

    if not isinstance(quality, dict):
        raise TypeError("'quality' must be a dictionary.")

    null_percent = quality.get("null_percen")
    unique_percent = quality.get("unique_percen")

    # Build the comment
    parts = [
        description,
        f"Type: {column_type}",
    ]

    if is_pk:
        parts.append("PK")

    if is_fk:
        parts.append("FK")

    if "nested_structures" in column_metadata:
        parts.append("Nested Structures")

    if null_percent is not None:
        parts.append(f"NP: {null_percent}")

    if unique_percent is not None:
        parts.append(f"UP: {unique_percent}")

    if is_fk and "referential_integrity" in quality:
        parts.append(f"RI: {quality['referential_integrity']}")

    return ". ".join(parts)

def generate_table_comment(table_metadata: dict) -> str:
    """
    Generate a standardized table comment from metadata.

    The generated comment follows this structure:

        Description.
        Upstream: <origin>.
        SourceFormat: <format>.
        Freshness: <update_frequency>.
        Grain: <grain>.

    Optional sections are omitted when their corresponding
    values are not provided.

    Args:
        table_metadata:
            Dictionary containing table metadata.

    Returns:
        A formatted table comment.
    """

    if not isinstance(table_metadata, dict):
        raise TypeError("table_metadata must be a dictionary.")

    description = table_metadata.get("description", "")
    if not isinstance(description, str):
        raise TypeError("'description' must be a string.")

    source = table_metadata.get("source", {})
    if not isinstance(source, dict):
        raise TypeError("'source' must be a dictionary.")

    data = source.get("data", {})
    if not isinstance(data, dict):
        raise TypeError("'source.data' must be a dictionary.")

    origin = source.get("origin")
    source_format = data.get("format")
    update_frequency = source.get("update_frequency")
    grain = table_metadata.get("grain")

    parts = []

    if description.strip():
        parts.append(description.strip().rstrip("."))

    if origin:
        parts.append(f"Upstream: {origin}")

    if source_format:
        parts.append(f"SourceFormat: {source_format}")

    if update_frequency:
        parts.append(f"Freshness: {update_frequency}")

    if grain:
        parts.append(f"Grain: {grain}")

    return ". ".join(parts)


In [ ]:
import yaml


def load_yaml_as_dict(file_path: str) -> dict:
    """
    Load a YAML file and return its contents as a dictionary.

    Args:
        file_path: Path to the YAML file.

    Returns:
        The YAML contents as a dictionary.
    """
    with open(file_path, "r", encoding="utf-8") as file:
        data = yaml.safe_load(file)

    if data is None:
        return dict()

    if not isinstance(data, dict):
        raise TypeError("The YAML root element must be a dictionary.")

    return data


In [ ]:
def get_table_metadata_path(metadata_type, quality, platform, lvl, table):
    # gets the base param volume for that table
    # metadata_type must be either schema or metadata
    if metadata_type == "schema":
        file_name = "schema.yml"
    elif metadata_type == "metadata":
        file_name = "metadata.yml"
    else:
        raise ValueError("metadata_type argument can be either 'schema' or 'metadata'")
    #folder = dbutils.widgets.get('comments_volume')
    folder = "/Volumes/oxygen_dev/landing/sharepoint_test/metadata/comments/"
    file_path = Path(folder) / str(quality) / str(platform) / str(table) / str(lvl) / file_name
    return file_path

In [ ]:
def update_table_process(catalog, schema, table, quality, platform, lvl, table_name):

    volume_table_metadata = get_table_metadata_path("metadata", quality, platform, lvl, table_name)
    volume_columns_metadata = get_table_metadata_path("schema", quality, platform, lvl, table_name)

    tab_meta = load_yaml_as_dict(volume_table_metadata)
    cols_meta = load_yaml_as_dict(volume_columns_metadata)

    tab_comment = generate_table_comment(tab_meta)
    cols_comments = dict()
    for c, meta in cols_meta.items():
        cols_comments[c] = generate_column_comment(meta)

    quality_description = {
        "bronze": "Indicates that the data is as close as possible to raw, with no more alteration than data type specification and shaping (for example, from JSON to Table)",
        "silver": "The data is close to bronze in terms of shaping, but some quality transformations, filtering row level column calculations and little restructuring has been applied (for example an array from a JSON has been exploded into a table).",
        "gold": "The data is already prepared for consuming, some aggregations, or joins might have been applied or some extra filters."
    }

    lvl_description = {
        "raw": "Indicates that the data has been loaded from origin, and transformed into a table with one singular column 'payload' containing the original JSON list unpacked. Only bronze has this level.",
        "base": "Indicates that it has the data from raw level but the top level keys have been unpacked. Only bronze has this level.",
        "current": "Indicates that it is the base level but with the latest snapshot. Only bronze has this level.",
        "history": "Indicates that it is part of the base level, keeping some of the columns. Only bronze has this level.",
        "cleaned": "Indicates that the data from current has been cleaned, filtered and new columns have been added. Only silver has this level.",
        "fact": "Indicates that this is a Fact table. Only gold has this level.",
        "dim": "Indicates that this is a Dimension table. Only gold has this level.",
        "agg": "indicates that this table is an aggregation. Only gold has this level."
    }

    update_table_metadata(catalog, schema, table, tab_comment, {"quality_description": quality_description.get(quality, None), "level_description": lvl_description.get(lvl, None)}, cols_comments)
    return None

In [ ]:

qualities = ["bronze", "silver"]
platforms = ["whoz", "analytic", "perso"]
schemas = [f"{quality}_{platform}" for quality in qualities for platform in platforms]
table_lvl = ["raw", "base", "current", "history", "cleaned", "fact"]
platform_tables = {
    "whoz": ["accreditation", "certification", "skill", "user", "talent", "profile"],
    "analytic": ["bu", "service_line", "entity", "society", "department", "site"],
    "perso": ["collab_status", "leave", "workers"]
}

quality_lvls = {
    "bronze": ["raw", "base", "current", "history"],
    "silver": ["cleaned"],
    "gold": ["fact"]
}

extra_tables = ["profile_educations", "profile_positions", "profile_aptitudes", "certification_aptitudes", "accreditation_aptitudes", "dim_organization", "dim_date", "collab_referential", "_user", "_talent"]

schema_tables = list()
for q in qualities:
    for p, ts in platform_tables.items():
        for t in ts:
            for lvl in quality_lvls[q]:
                schema_tables.append((f"{q}_{p}", f"{lvl}_{t}", q, p, lvl, t))

extra = [
    ("silver_whoz", "profile_educations", "silver", "whoz", "cleaned", "profile_educations"),
    ("silver_whoz", "profile_positions", "silver", "whoz", "cleaned", "profile_positions"),
    ("silver_whoz", "profile_aptitudes", "silver", "whoz", "cleaned", "profile_aptitudes"),
    ("silver_whoz", "certification_aptitudes", "silver", "whoz", "cleaned", "certification_aptitudes"),
    ("silver_whoz", "accreditation_aptitudes", "silver", "whoz", "cleaned", "accreditation_aptitudes"),
    ("gold_analytic", "dim_organization", "gold", "analytic", "dim", "organization"),
    ("gold_date", "dim_date", "gold", "date", "dim", "date"),
    ("gold_perso", "collab_referential", "gold", "perso", "fact", "collab_referential"),
    ("gold_whoz", "_user", "gold", "whoz", "fact", "_user"),
    ("gold_whoz", "_talent", "gold", "whoz", "fact", "_talent")
]

all_tables = schema_tables + extra

CATALOG = "oxygen_dev"
for t in all_tables:
    update_table_process(CATALOG, *t)